# Practical PyTorch · Part 7 — Companion Notebook

Your first capstone: rebuild the MLP from Parts 5–6, load **trained weights we made for you**, and wrap it in a Gradio app that reads handwritten digits. No training here — Phase II covers that.

In [ ]:
!pip install -q gradio

## 1. Rebuild the exact network (same as Part 5)

In [ ]:
import torch
import torch.nn as nn

mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

## 2. Load the trained weights

Downloaded once from the companion repo, then poured into your model with `load_state_dict`.

In [ ]:
url = "https://github.com/engineers-musings/blog-companion/raw/main/running-models-foundations/weights/mnist_mlp.pt"
state = torch.hub.load_state_dict_from_url(url, map_location="cpu")
mlp.load_state_dict(state)
mlp.eval()

## 3. The model as a function

Preprocess the drawing the same way the training data was prepared: grayscale, 28×28, inverted (MNIST is white-on-black), normalized with MNIST's mean/std.

In [ ]:
from PIL import Image, ImageOps
import torch.nn.functional as F

def read_digit(img):
    if img is None:
        return {}
    img = img.convert("L").resize((28, 28))
    img = ImageOps.invert(img)
    x = torch.tensor([list(img.getdata())], dtype=torch.float32) / 255.0
    x = (x - 0.1307) / 0.3081
    with torch.no_grad():
        probs = F.softmax(mlp(x), dim=1)[0]
    return {str(i): float(probs[i]) for i in range(10)}

## 4. Put a face on it with Gradio

If the sketchpad hands `read_digit` a dict instead of an image, pull the picture out with `img = img["composite"]`.

In [ ]:
import gradio as gr

gr.Interface(
    fn=read_digit,
    inputs=gr.Sketchpad(type="pil", image_mode="L"),
    outputs=gr.Label(num_top_classes=3),
    title="Draw a digit, the MLP reads it",
    description="Scribble a 0–9 and the network you built guesses what it is.",
).launch(share=True)

**Next: Part 8 — Teaching a Model to See: Convolution and LeNet.**